In [1]:
import os
import json
from tqdm import tqdm

import numpy as np
import pandas as pd
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

import torch
from transformers import (AutoConfig, AutoModelForTokenClassification, AutoTokenizer,
                          DataCollatorForTokenClassification, HfArgumentParser, PretrainedConfig,
                          PreTrainedTokenizerFast, Trainer, TrainingArguments, set_seed,)
from peft import (LoraConfig, get_peft_model, TaskType,
                  PeftModel, PeftConfig)

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level = logging.INFO)
transformers_logger = logging.getLogger("transformers")
transformers_logger.setLevel(logging.WARNING)

from bayartsogtya_utils import dataset_prep

In [2]:
class CFG:
    wandb = True
    report_to = None
    lab_assignment = 3
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    output_dir = "processed_data"
    model_save_dir = 'roberta-base-ner-demo'

    tokenizer_name = 'bayartsogt/mongolian-roberta-base'
    model_name = 'bayartsogt/mongolian-roberta-base'

    project = 'NUM-Machine-Learning-Lab-4'
    name = "Lab 4 Model Fine-Tuning - Mongolian Roberta NER - PEFT Train, 10 Epochs"

    config = {
        "output_dir": "mn_roberta_lab4_finetune_PEFT",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-3,
        'num_train_epochs': 10,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True
    }

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

# 1. Data prep

In [3]:
with open('./raw/NER_v1.0.json', 'r') as reader:
    lines = reader.readlines()
lines = [json.loads(x) for x in lines]

raw_dataset, labels, label2idx, idx2label, num_labels = dataset_prep(lines)
raw_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 10162
})

In [4]:
text_column_name = 'tokens'
label_column_name = 'ner_tags'
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name, use_fast = True, add_prefix_space = True)

# Tokenize all texts and align the labels with them.
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples[text_column_name],
                                 padding = "max_length",
                                 truncation = True,
                                 max_length = 128, 
                                 is_split_into_words = True)
    labels = []
    for i, label in enumerate(examples[label_column_name]):
        word_ids = tokenized_inputs.word_ids(batch_index = i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_ids.append(label[word_idx])

            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels

    return tokenized_inputs

tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched = True, num_proc = CFG.num_workers, desc = "Running tokenizer on train dataset")
all_dataset = tokenized_dataset.train_test_split(test_size = CFG.test_size)
all_dataset

Running tokenizer on train dataset (num_proc=12):   0%|          | 0/10162 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 8129
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2033
    })
})

# 2. Evaluation metric

In [5]:
metric = load_metric(CFG.eval_metric)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions = true_predictions, references = true_labels)
    
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# 3. Training

In [6]:
auto_config = AutoConfig.from_pretrained(CFG.model_name, 
                                         num_labels = num_labels,
                                         finetuning_task = config['finetuning_task'])

model = AutoModelForTokenClassification.from_pretrained(CFG.model_name, config = auto_config)
data_collator = DataCollatorForTokenClassification(tokenizer, pad_to_multiple_of = None)

model.config.label2id = label2idx
model.config.id2label = idx2label

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at bayartsogt/mongolian-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [7]:
test_ver = 2
OUTPUT_MODEL = os.path.join(CFG.model_save_dir, f"test_v{test_ver}")

training_args = TrainingArguments(
    report_to = CFG.report_to,
    output_dir = OUTPUT_MODEL,
    num_train_epochs = config["num_train_epochs"],
    per_device_train_batch_size = config["train_batch_size"],
    per_device_eval_batch_size = config["eval_batch_size"],
    overwrite_output_dir = config["overwrite_output_dir"],
    learning_rate = config["learning_rate"],
    weight_decay = config["weight_decay"],
    evaluation_strategy = config["evaluation_strategy"],
    do_eval = True,
    disable_tqdm = False
)

In [8]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = all_dataset['train'],
    eval_dataset = all_dataset['test'],
    tokenizer = tokenizer,
    data_collator = data_collator,
    compute_metrics = compute_metrics,
)

In [9]:
trainer.train()

  0%|          | 0/2550 [00:00<?, ?it/s]

  0%|          | 0/64 [00:00<?, ?it/s]

{'eval_loss': 0.0858672559261322, 'eval_precision': 0.8778580024067388, 'eval_recall': 0.9022881880024737, 'eval_f1': 0.8899054589813967, 'eval_accuracy': 0.9728065284606416, 'eval_runtime': 20.3526, 'eval_samples_per_second': 99.889, 'eval_steps_per_second': 3.145, 'epoch': 1.0}
{'loss': 0.1246, 'grad_norm': 2.2643613815307617, 'learning_rate': 1.607843137254902e-05, 'epoch': 1.96}


  0%|          | 0/64 [00:00<?, ?it/s]

{'eval_loss': 0.08124353736639023, 'eval_precision': 0.90520063037944, 'eval_recall': 0.9235621521335807, 'eval_f1': 0.9142892126851966, 'eval_accuracy': 0.9776471184550661, 'eval_runtime': 20.7599, 'eval_samples_per_second': 97.929, 'eval_steps_per_second': 3.083, 'epoch': 2.0}


...

  0%|          | 0/64 [00:00<?, ?it/s]

{'eval_loss': 0.12835411727428436, 'eval_precision': 0.9189684673673918, 'eval_recall': 0.9299938157081015, 'eval_f1': 0.9244482695026741, 'eval_accuracy': 0.9785087941608799, 'eval_runtime': 20.3133, 'eval_samples_per_second': 100.082, 'eval_steps_per_second': 3.151, 'epoch': 9.0}
{'loss': 0.0032, 'grad_norm': 0.07554082572460175, 'learning_rate': 3.921568627450981e-07, 'epoch': 9.8}


  0%|          | 0/64 [00:00<?, ?it/s]

{'eval_loss': 0.13119731843471527, 'eval_precision': 0.917796506656895, 'eval_recall': 0.9293753865182437, 'eval_f1': 0.9235496558505408, 'eval_accuracy': 0.9782807035328704, 'eval_runtime': 20.604, 'eval_samples_per_second': 98.67, 'eval_steps_per_second': 3.106, 'epoch': 10.0}
{'train_runtime': 766.6015, 'train_samples_per_second': 106.039, 'train_steps_per_second': 3.326, 'train_loss': 0.03376127944857466, 'epoch': 10.0}


TrainOutput(global_step=2550, training_loss=0.03376127944857466, metrics={'train_runtime': 766.6015, 'train_samples_per_second': 106.039, 'train_steps_per_second': 3.326, 'train_loss': 0.03376127944857466, 'epoch': 10.0})